In [1]:
pwd

'c:\\Users\\ADMIN\\Documents\\projects\\AIFFEL_quest_eng\\NLP\\NLP01'

In [2]:
import os
import re
from collections import Counter

import urllib.request
import zipfile

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import tarfile


In [3]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)


DEVICE: cuda


In [4]:
# 1) 데이터 폴더
DATA_DIR = "data_korean_english_news_v1"
os.makedirs(DATA_DIR, exist_ok=True)

# 2) 저장소 zip URL
ZIP_URL = "https://github.com/jungyeul/korean-parallel-corpora/archive/refs/heads/master.zip"  # [web:1]
ZIP_PATH = os.path.join(DATA_DIR, "korean-parallel-corpora-master.zip")

# 3) 다운로드
if not os.path.exists(ZIP_PATH):
    print("Downloading corpus zip...")
    urllib.request.urlretrieve(ZIP_URL, ZIP_PATH)
    print("Download finished.")

# 4) zip 압축 해제 (저장소 디렉토리 생성)
EXTRACT_ROOT = os.path.join(DATA_DIR, "korean-parallel-corpora-master")
if not os.path.exists(EXTRACT_ROOT):
    print("Extracting zip...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(DATA_DIR)
    print("Extraction finished.")
else:
    print("Extract root already exists:", EXTRACT_ROOT)

# 5) korean-english-news-v1 디렉토리
BASE_DIR = EXTRACT_ROOT  # = data_korean_english_news_v1/korean-parallel-corpora-master
NEWS_DIR = os.path.join(BASE_DIR, "korean-english-news-v1")
print("NEWS_DIR:", NEWS_DIR)
print("FILES before tar extraction:", os.listdir(NEWS_DIR))

# 6) korean-english-park.*.tar.gz 모두 압축 해제
for fname in os.listdir(NEWS_DIR):
    if fname.endswith(".tar.gz"):
        tar_path = os.path.join(NEWS_DIR, fname)
        print("Extracting:", tar_path)
        with tarfile.open(tar_path, "r:gz") as tar:
            tar.extractall(NEWS_DIR)
        print("Done:", fname)

print("FILES after tar extraction:", os.listdir(NEWS_DIR))


Extract root already exists: data_korean_english_news_v1\korean-parallel-corpora-master
NEWS_DIR: data_korean_english_news_v1\korean-parallel-corpora-master\korean-english-news-v1
FILES before tar extraction: ['korean-english-park.dev.en', 'korean-english-park.dev.ko', 'korean-english-park.dev.tar.gz', 'korean-english-park.test.en', 'korean-english-park.test.ko', 'korean-english-park.test.tar.gz', 'korean-english-park.train.en', 'korean-english-park.train.ko', 'korean-english-park.train.tar.gz', 'README.md']
Extracting: data_korean_english_news_v1\korean-parallel-corpora-master\korean-english-news-v1\korean-english-park.dev.tar.gz
Done: korean-english-park.dev.tar.gz
Extracting: data_korean_english_news_v1\korean-parallel-corpora-master\korean-english-news-v1\korean-english-park.test.tar.gz
Done: korean-english-park.test.tar.gz
Extracting: data_korean_english_news_v1\korean-parallel-corpora-master\korean-english-news-v1\korean-english-park.train.tar.gz
Done: korean-english-park.train.t

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_23432\822202083.py:37: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(NEWS_DIR)


In [ ]:
# 1. 데이터셋의 공통 접두어(Prefix) 설정
# 데이터 파일들이 저장된 디렉토리 내에서 공통적으로 사용하는 파일 이름의 기반이 되는 문자열입니다.
TRAIN_BASE_PATH = "korean-english-park.train"

# 2. 한국어(KO) 학습 데이터의 전체 경로 구성
# os.path.join을 사용하여 운영체제에 독립적인 경로를 생성합니다.
# NEWS_DIR 상위 디렉토리 하위에 위치한 한국어 데이터셋의 절대/상대 경로를 정의합니다.
KO_TRAIN_PATH = os.path.join(NEWS_DIR, "korean-english-park.train.ko")

# 3. 영어(EN) 학습 데이터의 전체 경로 구성
# 위와 동일하게 NEWS_DIR 내에 위치한 영어 데이터셋의 경로를 정의합니다.
# 일반적으로 병렬 코퍼스는 행(Line) 단위로 1:1 매칭되어 관리되므로, 두 경로의 무결성이 중요합니다.
EN_TRAIN_PATH = os.path.join(NEWS_DIR, "korean-english-park.train.en")

In [ ]:
# --- 하이퍼파라미터 설정 (Hyperparameters) ---
# 모델 입력의 최대 시퀀스 길이. 이를 초과하는 문장은 절단(Truncation)하거나 패딩(Padding) 처리를 결정하는 기준입니다.
MAX_LEN = 40
# 단어 집합(Vocabulary)에 포함되기 위한 최소 출현 빈도. 희소 단어(Rare words)를 제거하여 모델의 노이즈를 줄입니다.
MIN_FREQ = 2
# 메모리 효율과 연산 속도를 고려하여 제한한 최대 단어 집합 크기입니다.
MAX_VOCAB = 20000

def preprocess_en(text: str) -> str:
    """
    영어 텍스트 전처리: 소문자 변환, 특수문자 정제 및 구두점 분리
    """
    # 1. 대소문자 구분을 없애 단어 집합의 크기를 최적화합니다.
    text = text.lower()
    # 2. 알파벳, 숫자, 기본 구두점을 제외한 모든 불필요한 노이즈 문자를 공백으로 치환합니다.
    text = re.sub(r"[^a-z0-9,.!?\' ]", " ", text)
    # 3. 구두점(,.!?)을 토큰으로 인식시키기 위해 앞뒤에 공백을 삽입하는 'Padding Punctuation' 작업을 수행합니다.
    text = re.sub(r"([,.!?])", r" \1 ", text)
    # 4. 중복된 공백을 단일 공백으로 통합하고 문장 앞뒤의 불필요한 공백을 제거합니다.
    text = re.sub(r"\s+", " ", text).strip()
    return text

def preprocess_ko(text: str) -> str:
    """
    한국어 텍스트 전처리: 한글 및 숫자 추출 및 구두점 분리
    """
    # 1. 한글 음절(가-힣), 숫자, 구두점만 남기고 나머지(한자, 특수기호 등)는 정제합니다.
    text = re.sub(r"[^가-힣0-9,.!?\' ]", " ", text)
    # 2. 한국어 문장 부호 역시 별도의 토큰으로 취급하기 위해 공백을 추가합니다.
    text = re.sub(r"([,.!?])", r" \1 ", text)
    # 3. 연속된 공백(White space)을 정규화하여 토큰화 효율을 높입니다.
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize_ko(text: str):
    """
    한국어 토큰화: 공백 기반 어절 단위 분리 (White-space Tokenization)
    참고: 한국어 특성상 형태소 분석이 권장되나, 여기서는 기본적인 어절 단위를 채택함.
    """
    return text.split()

def tokenize_en(text: str):
    """
    영어 토큰화: 공백 기반 단어 단위 분리
    """
    return text.split()

In [ ]:
# 1. 컨텍스트 매니저(with문)를 활용한 다중 파일 안전 열기
# 한국어와 영어 병렬 코퍼스를 utf-8 인코딩으로 동시에 로드합니다.
# 'with' 구문은 작업 완료 후 파일 디스크립터를 자동으로 해제하여 메모리 누수를 방지합니다.
with open(KO_TRAIN_PATH, encoding="utf-8") as f_ko, \
     open(EN_TRAIN_PATH, encoding="utf-8") as f_en:
    
    # 2. 리스트 컴프리헨션을 이용한 데이터 정제
    # 각 행(Line)의 양끝 공백 및 줄바꿈 문자를 제거(strip)하여 리스트에 저장합니다.
    kor_lines = [l.strip() for l in f_ko]
    eng_lines = [l.strip() for l in f_en]

# 3. 데이터 무결성 검증 (Assertion)
# 번역 쌍(Pair)이 1:1로 정확히 매칭되는지 확인합니다.
# 만약 두 파일의 라인 수가 다르다면 데이터 소스에 결함이 있는 것으로 판단하고 중단합니다.
assert len(kor_lines) == len(eng_lines)

# 4. 고유 데이터 추출 및 중복 제거
# zip을 통해 양국어 문장을 튜플 쌍으로 묶은 후, set 자료구조를 활용해 중복된 문장 쌍을 필터링합니다.
# 이는 모델이 동일한 데이터에 과적합(Overfitting)되는 것을 방지하기 위한 필수적인 정제 단계입니다.
pairs = list(set(zip(kor_lines, eng_lines)))  

# 5. 최종 데이터 규모 확인
# 정제 작업이 완료된 후 최종적으로 학습에 투입될 유효 샘플 수를 출력합니다.
print("총 문장 쌍 수:", len(pairs))

총 문장 쌍 수: 78941


In [ ]:
# 1. 최종 결과물을 저장할 리스트 초기화
# 토큰화가 완료된 한국어 및 영어 문장 리스트를 각각 담기 위한 버퍼입니다.
kor_tokenized = []
eng_tokenized = []

# 2. 데이터 반복 처리 (Iteration)
# 중복 제거된 병렬 문장 쌍(pairs)을 순회하며 개별 전처리를 수행합니다.
for ko, en in pairs:
    # 2-1. 언어별 전처리 함수 적용
    # 앞서 정의한 정규표현식 기반 로직을 통해 특수문자 제거 및 노이즈 정제를 진행합니다.
    ko_p = preprocess_ko(ko)
    en_p = preprocess_en(en)
    
    # 2-2. 빈 문장 필터링 (Exception Handling)
    # 전처리 결과 문장이 비어 있거나 유효하지 않은 경우, 학습 데이터에서 제외(skip)합니다.
    if not ko_p or not en_p:
        continue

    # 2-3. 토큰화 실행
    # 정제된 텍스트를 어절 또는 단어 단위의 토큰 리스트로 분할합니다.
    ko_tokens = tokenize_ko(ko_p)
    en_tokens = tokenize_en(en_p)

    # 2-4. 문장 길이 기반 필터링 (Length Constraint)
    # 모델의 최대 입력 길이(MAX_LEN)를 초과하지 않는 유효 데이터만 최종 선택합니다.
    # 이는 메모리 효율을 최적화하고 학습 시 과도한 패딩(Padding) 발생을 억제합니다.
    if len(ko_tokens) <= MAX_LEN and len(en_tokens) <= MAX_LEN:
        kor_tokenized.append(ko_tokens)
        eng_tokenized.append(en_tokens)

# 3. 결과 요약 및 데이터 샘플 확인
# 최종적으로 확보된 데이터 규모와 토큰화 품질을 검증하기 위한 디버깅용 출력입니다.
print("전처리 후 문장 쌍 수:", len(kor_tokenized))
print("예시 한국어 토큰:", kor_tokenized[0][:10])
print("예시 영어 토큰:", eng_tokenized[0][:10])

전처리 후 문장 쌍 수: 71902
예시 한국어 토큰: ['강간', '혐의로', '25년간', '수감생활을', '했던', '제리', '밀러', '48', '의', '무죄가']
예시 영어 토큰: ['a', 'man', 'who', 'spent', '25', 'years', 'in', 'prison', 'for', 'rape']


In [ ]:
# 1. 특수 토큰(Special Tokens) 정의
# <pad>: 문장 길이를 맞추기 위한 패딩
# <sos>: 문장 시작(Start of Sentence) 알림
# <eos>: 문장 종료(End of Sentence) 알림
# <unk>: 사전에 없는 단어(Unknown) 처리용
SPECIAL_TOKENS = ["<pad>", "<sos>", "<eos>", "<unk>"]

def build_vocab(corpus, min_freq=2, max_vocab=20000):
    """
    코퍼스로부터 빈도수 기반의 단어 사전을 생성하는 함수
    """
    # 2. 단어 빈도수 계산 (Word Frequency Counting)
    # 중첩 리스트 형태인 코퍼스를 순회하며 각 토큰의 출현 횟수를 카운트합니다.
    counter = Counter(token for sent in corpus for token in sent)
    
    # 3. 사전(Vocabulary) 리스트 초기화
    # 특수 토큰을 항상 인덱스 0~3번에 배치하여 고정된 역할을 부여합니다.
    vocab = SPECIAL_TOKENS.copy()
    
    # 4. 상위 빈도 단어 선별 및 추가
    # 빈도가 min_freq 이상인 단어 중, 빈도순 상위 max_vocab개만큼만 사전에 포함시킵니다.
    # 이를 통해 데이터 내 노이즈(오타 등)를 제거하고 모델의 파라미터 크기를 제어합니다.
    vocab += [w for w, c in counter.most_common(max_vocab) if c >= min_freq]
    
    # 5. 매핑 딕셔너리 생성 (Bidirectional Mapping)
    # stoi (String-to-Index): 단어를 숫자로 변환 (학습 시 사용)
    stoi = {w: i for i, w in enumerate(vocab)}
    # itos (Index-to-String): 숫자를 단어로 변환 (추론/결과 확인 시 사용)
    itos = {i: w for w, i in stoi.items()}
    
    return stoi, itos

In [ ]:
# 1. 소스(Source) 및 타겟(Target) 언어별 단어 사전 구축
# 한국어(src)와 영어(tgt) 코퍼스에 대해 각각 독립적인 Vocabulary 공간을 생성합니다.
# 하이퍼파라미터인 MIN_FREQ와 MAX_VOCAB을 적용하여 빈도수 기반 정제를 수행합니다.
src_stoi, src_itos = build_vocab(kor_tokenized, MIN_FREQ, MAX_VOCAB)
tgt_stoi, tgt_itos = build_vocab(eng_tokenized, MIN_FREQ, MAX_VOCAB)

# 2. 모델 제어를 위한 핵심 특수 토큰 인덱스 추출
# 2-1. PAD_IDX: 문장 길이를 맞추기 위한 패딩 인덱스입니다.
# 학습 시 Loss 계산에서 제외(Ignore index)하거나 Attention Mask를 생성할 때 참조합니다.
PAD_IDX_SRC = src_stoi["<pad>"]
PAD_IDX_TGT = tgt_stoi["<pad>"]

# 2-2. SOS/EOS_IDX: 문장의 시작과 끝을 나타내는 인덱스입니다.
# 디코더(Decoder)의 첫 입력값(<sos>)과 생성 종료 조건(<eos>)을 제어하기 위해 사용합니다.
SOS_IDX_TGT = tgt_stoi["<sos>"]
EOS_IDX_TGT = tgt_stoi["<eos>"]

# 3. 사전 구축 결과 검증
# 최종적으로 생성된 각 언어별 단어 사전의 크기를 확인합니다.
# 이 수치는 모델의 Embedding 레이어와 최종 Linear 레이어의 차원(Dimension)을 결정합니다.
print("SRC vocab size:", len(src_stoi))
print("TGT vocab size:", len(tgt_stoi))

SRC vocab size: 20004
TGT vocab size: 20004


In [ ]:
def encode(tokens, stoi):
    """
    텍스트 토큰 리스트를 고유 인덱스 시퀀스로 변환하고, 특수 토큰을 부착하는 함수
    """
    # 1. Unknown 토큰 처리 (OOV handling)
    # 단어 사전에 없는 단어가 들어올 경우 stoi.get의 default 인자로 <unk> 인덱스를 반환하여 
    # 런타임 에러를 방지하고 모델의 견고성(Robustness)을 확보합니다.
    ids = [stoi.get(t, stoi["<unk>"]) for t in tokens]
    
    # 2. 문장의 시작(Start)과 끝(End) 표식 추가
    # 시퀀스 양 끝에 <sos>와 <eos> 인덱스를 결합하여, 모델이 문장의 경계를 
    # 명확히 학습(Sentence boundary detection)할 수 있도록 구조화합니다.
    return [stoi["<sos>"]] + ids + [stoi["<eos>"]]

def pad_sequence(seq, max_len, pad_idx):
    """
    가변 길이의 시퀀스를 고정된 길이(max_len)로 맞추기 위한 패딩 함수
    """
    # 3. 제로 패딩(Zero-padding) 전략 수행
    # 현재 시퀀스 길이와 목표 최대 길이 간의 차이만큼 pad_idx를 후행(Trailing) 추가합니다.
    # 이는 배치(Batch) 연산 시 모든 데이터의 차원을 일치시켜 병렬 연산 효율을 극대화하기 위함입니다.
    return seq + [pad_idx] * (max_len - len(seq))

In [ ]:
class TranslationDataset(Dataset):
    """
    병렬 코퍼스를 모델 입력용 텐서로 변환하는 Custom Dataset 클래스
    """
    def __init__(self, src_corpus, tgt_corpus, src_stoi, tgt_stoi, max_len=MAX_LEN+2):
        # 1. 데이터셋 초기화
        # 소스/타겟 코퍼스와 각각의 단어 사전(stoi)을 매핑합니다.
        # max_len은 <sos>, <eos> 토큰이 추가된 것을 고려하여 기본값에 +2를 적용했습니다.
        self.src = src_corpus
        self.tgt = tgt_corpus
        self.src_stoi = src_stoi
        self.tgt_stoi = tgt_stoi
        self.max_len = max_len

    def __len__(self):
        # 2. 데이터셋 크기 반환
        # 전체 문장 쌍의 개수를 반환하여 DataLoader가 에포크당 스텝 수를 계산하게 합니다.
        return len(self.src)

    def __getitem__(self, idx):
        # 3. 데이터 샘플 생성 및 변환 로직
        # 3-1. 수치화(Numericalization): 토큰 리스트를 인덱스 시퀀스로 변환하고 특수 토큰을 부착합니다.
        src_ids = encode(self.src[idx], self.src_stoi)
        tgt_ids = encode(self.tgt[idx], self.tgt_stoi)

        # 3-2. 길이 제한(Truncation): 혹시 모를 오버플로우를 방지하기 위해 최대 길이 내로 슬라이싱합니다.
        src_ids = src_ids[:self.max_len]
        tgt_ids = tgt_ids[:self.max_len]

        # 3-3. 패딩(Padding): 배치 내 연산을 위해 가변 길이 문장을 고정된 max_len으로 맞춥니다.
        src_padded = pad_sequence(src_ids, self.max_len, PAD_IDX_SRC)
        tgt_padded = pad_sequence(tgt_ids, self.max_len, PAD_IDX_TGT)

        # 3-4. 텐서화(Tensorization): 리스트를 PyTorch LongTensor로 변환하여 GPU 연산이 가능하게 합니다.
        return torch.tensor(src_padded), torch.tensor(tgt_padded)

In [ ]:
# 1. 커스텀 데이터셋 인스턴스 생성
# 전처리 및 토큰화가 완료된 양국어 코퍼스와 각각의 단어 사전(stoi)을 주입합니다.
# 이 단계에서 각 문장은 숫자로 변환되고, 동일한 길이(max_len)로 패딩될 준비를 마칩니다.
dataset = TranslationDataset(kor_tokenized, eng_tokenized, src_stoi, tgt_stoi)

# 2. 배치 크기(Batch Size) 결정
# 한 번의 반복(Iteration)에서 모델이 학습할 데이터 샘플의 개수를 설정합니다.
# 하드웨어(GPU Memory) 사양과 학습 안정성 사이의 트레이드오프를 고려하여 64로 정의했습니다.
BATCH_SIZE = 64

# 3. 데이터 로더(DataLoader) 구성
# 구축된 데이터셋을 바탕으로 미니배치를 생성하고, 데이터를 섞어서 모델에 공급합니다.
# shuffle=True: 매 에포크마다 데이터 순서를 무작위로 섞어 모델의 일반화 성능을 높이고 편향을 방지합니다.
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

In [ ]:
class Encoder(nn.Module):
    """
    소스 문장을 입력받아 고정된 크기의 은닉 상태(Hidden State)로 압축하는 인코더 모듈
    """
    def __init__(self, vocab_size, emb_dim, hid_dim, pad_idx):
        super().__init__()
        # 1. 임베딩 레이어 정의
        # 단어 인덱스를 고차원 밀집 벡터(Dense Vector)로 변환합니다.
        # padding_idx를 지정하여 <pad> 토큰의 임베딩이 항상 0을 유지하도록(학습 제외) 설정했습니다.
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        
        # 2. GRU 순환 신경망 정의
        # LSTM보다 파라미터가 적으면서도 장기 의존성(Long-term dependency) 문제를 효과적으로 해결합니다.
        # batch_first=True 설정으로 입력 텐서의 형태를 (Batch, Seq, Feature)로 유지합니다.
        self.gru = nn.GRU(emb_dim, hid_dim, batch_first=True)

    def forward(self, src):
        # 3. 순전파(Forward Propagation)
        # 3-1. Embedding: (Batch_Size, Seq_Len) -> (Batch_Size, Seq_Len, Emb_Dim)
        embedded = self.embedding(src)
        
        # 3-2. GRU 연산:
        # outputs: 모든 시점(Time-step)의 은닉 상태 (Batch_Size, Seq_Len, Hid_Dim)
        # hidden: 마지막 시점의 요약된 은닉 상태 (1, Batch_Size, Hid_Dim)
        outputs, hidden = self.gru(embedded)
        
        # 인코딩된 전체 정보(outputs)와 문맥 벡터(hidden)를 반환합니다.
        return outputs, hidden

In [ ]:
class LuongAttention(nn.Module):
    """
    Luong의 General Attention 메커니즘: 디코더 hidden state와 인코더 output 사이의 정렬(Alignment) 점수를 계산
    """
    def __init__(self, hid_dim):
        super().__init__()
        # 1. 학습 가능한 가중치 행렬 (Wa) 정의
        # General score 함수: score(h_t, h_s) = h_t^T * Wa * h_s 를 구현하기 위함입니다.
        self.linear = nn.Linear(hid_dim, hid_dim, bias=False)

    def forward(self, hidden, encoder_outputs, mask=None):
        # 2. 디코더의 마지막 레이어 hidden state 추출
        # hidden: (Num_Layers, Batch_Size, Hid_Dim) -> (Batch_Size, Hid_Dim)
        hidden = hidden[-1] 

        # 3. 어텐션 스코어(Attention Score) 계산
        # 3-1. self.linear(hidden): 디코더 상태에 가중치 Wa를 곱합니다. -> (B, H)
        # 3-2. .unsqueeze(2): 행렬 곱(bmm)을 위해 차원을 확장합니다. -> (B, H, 1)
        # 3-3. torch.bmm: 인코더 전체 출력과 디코더 상태의 배치 행렬 곱 수행. -> (B, S, 1)
        scores = torch.bmm(
            encoder_outputs,                         # (Batch_Size, Seq_Len, Hid_Dim)
            self.linear(hidden).unsqueeze(2)         # (Batch_Size, Hid_Dim, 1)
        ).squeeze(2)                                 # 최종 스코어: (Batch_Size, Seq_Len)

        # 4. 마스킹(Masking) 및 소프트맥스(Softmax)
        # 패딩 토큰(<pad>) 위치에 매우 낮은 값을 채워 어텐션 가중치에서 제외되도록 합니다.
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        # 문장 내 각 단어에 대한 중요도를 확률 분포로 변환합니다. (0~1 사이 값)
        attn_weights = torch.softmax(scores, dim=1)  # (Batch_Size, Seq_Len)

        # 5. 컨텍스트 벡터(Context Vector) 생성
        # 가중치와 인코더 출력을 가중 합(Weighted Sum)하여 현재 시점에 필요한 정보만 추출합니다.
        # (B, 1, S) * (B, S, H) -> (B, 1, H) -> (B, H)
        context = torch.bmm(
            attn_weights.unsqueeze(1),               # 가중치 차원 확장
            encoder_outputs                          # 소스 문장 정보
        ).squeeze(1) 

        return context, attn_weights

In [ ]:
class Decoder(nn.Module):
    """
    Attention 메커니즘을 활용하여 타겟 시퀀스를 생성하는 디코더 모듈
    """
    def __init__(self, vocab_size, emb_dim, hid_dim, pad_idx):
        super().__init__()
        # 1. 레이어 정의
        # 타겟 언어의 단어 인덱스를 벡터화합니다.
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        
        # 2. 어텐션 기반 GRU 설계
        # 입력 임베딩(emb_dim)과 어텐션 컨텍스트 벡터(hid_dim)를 합쳐서 입력으로 받습니다.
        self.gru = nn.GRU(emb_dim + hid_dim, hid_dim, batch_first=True)
        
        # 루옹 어텐션 인스턴스 초기화
        self.attn = LuongAttention(hid_dim)
        
        # 3. 출력 레이어
        # GRU 출력과 컨텍스트 벡터를 결합(Concatenate)하여 최종 단어 확률을 계산합니다.
        self.fc_out = nn.Linear(hid_dim * 2, vocab_size)

    def forward(self, input, hidden, encoder_outputs, mask=None):
        # 4. 순전파 로직 (단일 시점 처리)
        # input: (Batch_Size) -> embedded: (Batch_Size, 1, Emb_Dim)
        embedded = self.embedding(input).unsqueeze(1) 

        # 5. 어텐션 계산
        # 현재 디코더 상태(hidden)와 인코더 전체 출력(encoder_outputs) 사이의 가중치를 계산합니다.
        # context: (B, H), attn: (B, S)
        context, attn = self.attn(hidden, encoder_outputs, mask)
        context = context.unsqueeze(1)  # RNN 입력을 위해 차원 확장 (B, 1, H)

        # 6. GRU 입력 구성 (Input Feeding approach)
        # 임베딩 벡터와 어텐션 컨텍스트를 결합하여 현재 시점의 풍부한 정보를 주입합니다.
        rnn_input = torch.cat([embedded, context], dim=2)  # (Batch_Size, 1, Emb_Dim + Hid_Dim)
        
        # output: (B, 1, H), hidden: (1, B, H)
        output, hidden = self.gru(rnn_input, hidden)

        # 7. 최종 예측 (Logits 생성)
        # RNN 출력과 컨텍스트 벡터를 다시 결합하여 최종 단어 분류를 수행합니다.
        output = output.squeeze(1)  # (B, H)
        context = context.squeeze(1) # (B, H)
        
        # (B, H*2) -> (B, Vocab_Size)
        logits = self.fc_out(torch.cat([output, context], dim=1))
        
        return logits, hidden, attn

In [ ]:
class Seq2Seq(nn.Module):
    """
    Encoder와 Decoder를 결합하여 전체 Sequence-to-Sequence 파이프라인을 관리하는 메인 클래스
    """
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def create_mask(self, src):
        """
        소스 문장의 패딩(<pad>) 위치를 파악하여 어텐션 연산 시 제외할 마스크 생성
        결과 형태: (Batch_Size, Src_Seq_Len) / PAD 토큰 위치는 False(0)로 처리
        """
        return (src != PAD_IDX_SRC).to(self.device)

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        # 1. 초기 파라미터 및 출력 버퍼 설정
        batch_size = src.size(0)
        tgt_len = tgt.size(1) # 타겟 문장의 길이
        vocab_size = self.decoder.fc_out.out_features # 예측할 단어 집합 크기

        # 각 시점(step)의 예측값(logits)을 저장할 빈 텐서 생성
        outputs = torch.zeros(batch_size, tgt_len, vocab_size, device=self.device)

        # 2. 인코딩 단계
        # 소스 문장을 입력받아 문맥 정보(encoder_outputs)와 초기 은닉 상태(hidden)를 추출합니다.
        encoder_outputs, hidden = self.encoder(src)
        
        # 어텐션이 패딩을 무시하도록 소스 마스크 생성
        mask = self.create_mask(src)

        # 3. 디코딩 단계 시작
        # 디코더의 첫 번째 입력으로 타겟 문장의 첫 토큰(<sos>)을 설정합니다.
        input_tok = tgt[:, 0]

        # 타겟 문장의 길이만큼 반복하며 한 단어씩 생성
        for t in range(1, tgt_len):
            # 3-1. 현재 시점의 단어 예측
            # logits: (B, V), hidden: 디코더의 다음 상태, attn: 어텐션 가중치
            logits, hidden, attn = self.decoder(input_tok, hidden, encoder_outputs, mask)
            
            # 예측 결과를 결과 버퍼에 저장
            outputs[:, t, :] = logits

            # 3-2. Teacher Forcing 결정
            # 무작위 확률에 따라 다음 시점의 입력을 결정합니다.
            teacher_force = torch.rand(1).item() < teacher_forcing_ratio
            
            # 모델이 예측한 가장 확률이 높은 단어 인덱스 추출
            top1 = logits.argmax(1)
            
            # Teacher Forcing이면 실제 정답(tgt)을, 아니면 모델의 예측값(top1)을 다음 입력으로 사용
            input_tok = tgt[:, t] if teacher_force else top1

        return outputs

In [ ]:
# 1. 모델 하이퍼파라미터(Hyperparameters) 설정
# EMB_DIM: 단어를 표현할 밀집 벡터의 차원. 정보의 표현력을 결정합니다.
# HID_DIM: GRU 내부 상태 및 어텐션 연산에 사용될 은닉 상태의 차원입니다.
EMB_DIM = 256
HID_DIM = 512

# 2. 인코더 및 디코더 객체 생성
# 각 언어의 단어 사전 크기(vocab_size)와 패딩 인덱스를 주입하여 레이어를 구성합니다.
encoder = Encoder(len(src_stoi), EMB_DIM, HID_DIM, PAD_IDX_SRC)
decoder = Decoder(len(tgt_stoi), EMB_DIM, HID_DIM, PAD_IDX_TGT)

# 3. 전체 Seq2Seq 모델 통합 및 가속기(GPU/CPU) 배치
# 구축된 모듈들을 하나의 시스템으로 묶고, 지정된 연산 장치(DEVICE)로 메모리를 할당합니다.
model = Seq2Seq(encoder, decoder, DEVICE).to(DEVICE)

# 4. 손실 함수(Loss Function) 정의
# 다중 클래스 분류를 위한 CrossEntropyLoss를 사용합니다.
# ignore_index: 패딩 토큰(<pad>)에 대한 손실은 계산하지 않도록 설정하여 모델이 불필요한 학습을 하지 않게 합니다.
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX_TGT)

# 5. 최적화 알고리즘(Optimizer) 설정
# Adam: 학습률을 유동적으로 조절하여 수렴 속도가 빠르고 안정적인 최적화 도구입니다.
# lr=1e-3: 초기 학습률(Learning Rate)을 0.001로 설정하여 가중치 업데이트 강도를 조절합니다.
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
def train_one_epoch(model, dataloader, optimizer, criterion, device):
    """
    전체 데이터셋을 한 번 훑으며 모델의 가중치를 업데이트하는 단일 에포크 학습 함수
    """
    # 1. 모델을 학습 모드로 전환
    # Dropout이나 Batch Normalization 등 학습 시에만 필요한 동작을 활성화합니다.
    model.train()
    epoch_loss = 0

    # 2. 미니배치 단위 데이터 반복 로드
    for src, tgt in dataloader:
        # 데이터를 지정된 연산 장치(GPU/CPU)로 전송
        src, tgt = src.to(device), tgt.to(device)

        # 3. 그래디언트 초기화
        # 이전 배치에서 계산된 기울기가 누적되지 않도록 초기화합니다.
        optimizer.zero_grad()
        
        # 4. 순전파(Forward Pass) 수행
        # Teacher Forcing 확률을 0.5로 설정하여 학습의 안정성과 자생력을 동시에 도모합니다.
        # outputs 형태: (Batch_Size, Tgt_Len, Vocab_Size)
        outputs = model(src, tgt, teacher_forcing_ratio=0.5)

        # 5. 손실(Loss) 계산을 위한 데이터 형태 변환
        output_dim = outputs.shape[-1]
        
        # Loss 계산 시 <sos> 토큰(인덱스 0)은 제외하고 예측값과 정답지를 비교합니다.
        # CrossEntropyLoss 입력을 위해 3차원 텐서를 2차원으로 펼쳐주는(Reshape) 공정입니다.
        loss = criterion(
            outputs[:, 1:].reshape(-1, output_dim), # 예측: (B*(T-1), V)
            tgt[:, 1:].reshape(-1)                  # 정답: (B*(T-1))
        )
        
        # 6. 역전파(Backward Pass) 및 가중치 업데이트
        loss.backward()
        
        # 6-1. Gradient Clipping: 기울기 폭주(Exploding Gradients)를 방지하기 위해 
        # L2 노름 기준 임계값(1.0)을 넘지 않도록 기울기를 조정(Scaling)합니다.
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        
        # 6-2. 파라미터 업데이트: Optimizer를 통해 가중치를 갱신합니다.
        optimizer.step()

        # 누적 손실 기록
        epoch_loss += loss.item()

    # 에포크 당 평균 손실값 반환
    return epoch_loss / len(dataloader)

In [ ]:
# 1. 학습 반복 횟수(Epochs) 설정
# 전체 데이터셋을 총 몇 번 반복하여 학습할지 결정합니다.
# 현재는 5회로 설정되어 있으며, 손실값(Loss)의 수렴 추이에 따라 조정이 필요합니다.
N_EPOCHS = 5  

# 2. 에포크 루프 실행
# 1부터 N_EPOCHS까지 반복하며 모델의 가중치를 점진적으로 최적화합니다.
for epoch in range(1, N_EPOCHS + 1):
    
    # 3. 단일 에포크 학습 수행
    # 앞서 정의한 train_one_epoch 함수를 호출하여 가중치 업데이트 및 평균 손실을 계산합니다.
    # model, dataloader, optimizer 등 모든 컴포넌트가 이 시점에 유기적으로 작동합니다.
    train_loss = train_one_epoch(model, dataloader, optimizer, criterion, DEVICE)
    
    # 4. 학습 상태 모니터링
    # 현재 에포크 번호와 계산된 평균 손실(Train Loss)을 소수점 4자리까지 출력합니다.
    # Loss가 지속적으로 감소하는지 확인하여 모델의 학습 여부를 판별하는 지표가 됩니다.
    print(f"[Epoch {epoch}] Train Loss: {train_loss:.4f}")

In [20]:
N_EPOCHS = 5  # 필요시 늘리기

for epoch in range(1, N_EPOCHS + 1):
    train_loss = train_one_epoch(model, dataloader, optimizer, criterion, DEVICE)
    print(f"[Epoch {epoch}] Train Loss: {train_loss:.4f}")


[Epoch 1] Train Loss: 6.1854
[Epoch 2] Train Loss: 5.4167
[Epoch 3] Train Loss: 4.8492
[Epoch 4] Train Loss: 4.4130
[Epoch 5] Train Loss: 4.1194


In [ ]:
def greedy_decode(model, src_tokens, max_len=MAX_LEN+2):
    """
    학습된 모델을 사용하여 입력 문장을 번역하는 추론 함수 (Greedy Search 방식)
    """
    # 1. 평가 모드 전환 및 그래디언트 계산 비활성화
    # Dropout 등의 학습용 동작을 끄고 메모리 효율을 위해 연산 기록을 남기지 않습니다.
    model.eval()
    with torch.no_grad():
        # 2. 입력 데이터 전처리 및 텐서화
        # 소스 토큰을 인덱스로 변환, 길이 제한 및 패딩을 적용하여 배치 차원(unsqueeze)을 추가합니다.
        src_ids = encode(src_tokens, src_stoi)
        src_ids = src_ids[:max_len]
        src_padded = pad_sequence(src_ids, max_len, PAD_IDX_SRC)
        src_tensor = torch.tensor(src_padded).unsqueeze(0).to(DEVICE)

        # 3. 소스 문장 인코딩
        # 인코더를 통해 소스 문장의 문맥(encoder_outputs)과 초기 상태(hidden)를 확보합니다.
        encoder_outputs, hidden = model.encoder(src_tensor)
        mask = model.create_mask(src_tensor)

        # 4. 디코딩 시작 준비
        # 시작 토큰(<sos>)을 첫 입력으로 설정하고 결과를 담을 리스트를 초기화합니다.
        input_tok = torch.tensor([SOS_IDX_TGT], device=DEVICE)
        generated = []

        # 5. 순차적 단어 생성 루프
        for _ in range(max_len):
            # 5-1. 현재 시점 예측
            logits, hidden, attn = model.decoder(input_tok, hidden, encoder_outputs, mask)
            
            # 5-2. Greedy Selection: 가장 높은 확률을 가진 단어 인덱스를 선택
            top1 = logits.argmax(1)
            
            # 5-3. 종료 조건 확인: 모델이 문장 끝(<eos>)을 예측하면 생성을 중단합니다.
            if top1.item() == EOS_IDX_TGT:
                break
            
            generated.append(top1.item())
            
            # 5-4. 피드백: 예측한 단어를 다음 시점의 입력으로 사용
            input_tok = top1

    # 6. 인덱스를 다시 단어로 복원 (Detokenization)
    # 수치화된 데이터를 인간이 읽을 수 있는 언어로 변환하고 특수 토큰을 정제합니다.
    tokens = [tgt_itos.get(i, "<unk>") for i in generated]
    tokens = [t for t in tokens if t not in ["<pad>", "<sos>", "<eos>"]]
    return " ".join(tokens)

In [ ]:
def translate_ko2en(model, sentence_ko: str):
    """
    원시 한국어 문장을 입력받아 영어로 번역된 텍스트를 반환하는 통합 함수
    """
    # 1. 원문 텍스트 정제 (Normalization)
    # 정규표현식을 사용하여 한국어 이외의 노이즈 문자를 제거합니다.
    ko_p = preprocess_ko(sentence_ko)
    
    # 2. 토큰화 수행 (Tokenization)
    # 정제된 문장을 모델 학습 시와 동일한 어절 단위(White-space)로 분리합니다.
    ko_tokens = tokenize_ko(ko_p)
    
    # 3. 추론 엔진 실행 (Inference via Greedy Decoding)
    # 토큰화된 리스트를 모델과 사전(Vocab)을 통해 인덱스화하고, 
    # 오토레그레시브(Autoregressive) 방식으로 영어 문장을 생성하여 반환합니다.
    return greedy_decode(model, ko_tokens)

In [24]:
# 1. 테스트용 입력 문장 리스트 정의
# 모델의 일반화 능력을 확인하기 위해 일상적인 문장과 의문문, 설명문 등으로 구성합니다.
test_sentences = [
    "나는 오늘 아침을 먹었다.",
    "나의 이름은 김종하 입니다.",
    "걸음보다도 빠른 내 마음은"
]

# 2. 테스트 루프 실행
# 각 문장을 순회하며 모델의 예측 결과(Prediction)를 출력합니다.
for s in test_sentences:
    # 원문(Source) 출력
    print("SRC:", s)
    
    # 3. 통합 번역 함수 호출 및 결과 출력
    # 내부적으로 [전처리 -> 토큰화 -> 인코딩 -> 디코딩 -> 역토큰화] 과정을 거칩니다.
    # 모델의 추론(Inference) 능력을 확인하는 최종 단계입니다.
    print("PRED:", translate_ko2en(model, s))
    
    # 가독성을 위한 개행
    print()

SRC: 나는 오늘 아침을 먹었다.
PRED: i am like a today ,

SRC: 나의 이름은 김종하 입니다.
PRED: here's you have a <unk>

SRC: 걸음보다도 빠른 내 마음은
PRED: the my <unk> is my life .

